# Chapter 32 — Dynamic Measurement: When Strain Changes Fast
### *Modern Strain Gage Technology* — Companion Notebook

When strain changes fast, the signal carries information a static reading throws away — natural frequencies, mode shapes, damping — and the tools that read it out are the Fourier transform and the dynamic amplification factor.

Companion notebook to Chapter 32 — Dynamic Measurement: When Strain Changes Fast.

Static strain measurement assumes the structure is in equilibrium: the load is applied slowly, the strain field has fully developed, and the gage reading reflects the actual stress state. Dynamic measurement abandons that assumption. When loads change rapidly — vibration, impact, acoustic excitation, flutter — the strain signal carries frequency content that static measurements ignore, and that content carries information static measurements cannot extract: natural frequencies, mode shapes, damping ratios, and transient response characteristics.

The *Fourier transform* is the essential tool for dynamic strain analysis. Decomposing the time-domain strain record into its frequency components reveals which structural modes are contributing to the response. Each natural frequency appears as a peak in the amplitude spectrum; the relative amplitude of those peaks reflects the modal participation for the given excitation condition. A gage placed near a node of a particular mode will not detect that mode regardless of instrument sensitivity — this is a geometric constraint, not an instrument limitation. Gage placement in dynamic testing must therefore be informed by a prior understanding of the expected mode shapes.

*Dynamic amplification* is the second key concept: near resonance, the dynamic response can be many times larger than the equivalent static response. A structure whose static strain under a given load is 100 με may experience 5000 με under dynamic loading near its first natural frequency — a factor of 1/(2ζ) at resonance for a single-degree-of-freedom system. This amplification is what makes resonance both dangerous (unexpectedly large stresses in service) and useful (it amplifies small signals in sensitive diagnostic testing).

**This notebook demonstrates:**
1. **FFT modal identification** — three structural modes (47, 87.3, and 312.5 Hz) recovered from a simulated dynamic strain signal buried in broadband noise. This regenerates Figure 32.2.
2. **Dynamic Amplification Factor** — DAF curves for four damping ratios (ζ = 0.01, 0.05, 0.10, 0.20) showing the 1/(2ζ) resonance peak. This regenerates Figure 32.4.

In [ ]:
# Colab setup: uncomment to install dependencies
# !pip install numpy matplotlib

from pathlib import Path

import numpy as np
from numpy.fft import fft, fftfreq
import matplotlib.pyplot as plt

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# All figures are written here. The book includes these exact files, so the
# filenames must not change.
OUT_DIR = Path("../src/media/ch32")
OUT_DIR.mkdir(parents=True, exist_ok=True)

## Identifying Natural Frequencies from Dynamic Strain

When a structure vibrates, each natural (resonant) frequency leaves a distinct spike in the FFT of the strain signal. The simulated signal below superposes **three** modal contributions — 47, 87.3, and 312.5 Hz — on a static offset and broadband noise, representative of a real structure under operational loading. In the time domain these modes blur into a single irregular waveform; the frequency domain pulls them apart.

We build the single-sided *amplitude spectrum*: remove the DC offset, take the FFT, and double the positive-frequency magnitudes so that a pure sinusoid of amplitude *A* shows a peak of height *A*. The top panel shows the time-domain strain history; the bottom panel shows the spectrum, where each mode appears as a clear peak rising above the noise floor and its frequency can be read directly.

In [ ]:
# --- Simulated dynamic strain signal: three structural modes + broadband noise ---
SAMPLE_RATE_HZ = 5000.0            # acquisition sample rate, f_s
DURATION_S = 2.0                   # record length
DC_OFFSET_UE = 200.0               # static strain offset (microstrain)
NOISE_STD_UE = 5.0                 # broadband measurement noise (microstrain RMS)
RANDOM_SEED = 7                    # seed so the noise realization is reproducible

# Each excited mode as (frequency [Hz], amplitude [microstrain]).
MODES = [(47.0, 90.0), (87.3, 65.0), (312.5, 45.0)]

# Number of samples shown in the time-domain panel (first 400 ms).
TIME_PLOT_SAMPLES = int(0.4 * SAMPLE_RATE_HZ)


def single_sided_amplitude_spectrum(signal, sample_rate):
    """Return (frequencies, amplitude) for the single-sided FFT amplitude spectrum.

    The mean (DC) component is removed first. The two-sided FFT magnitude is
    then doubled over the positive-frequency half so that a pure sinusoid of
    amplitude A produces a spectral peak of height A.
    """
    n = len(signal)
    spectrum = fft(signal - signal.mean()) / n
    freqs = fftfreq(n, 1 / sample_rate)
    positive = freqs > 0
    return freqs[positive], 2 * np.abs(spectrum[positive])


np.random.seed(RANDOM_SEED)
t = np.arange(0, DURATION_S, 1 / SAMPLE_RATE_HZ)
strain = DC_OFFSET_UE + np.random.normal(0, NOISE_STD_UE, len(t))
for freq_hz, amp_ue in MODES:
    strain = strain + amp_ue * np.sin(2 * np.pi * freq_hz * t)

f_pos, mag = single_sided_amplitude_spectrum(strain, SAMPLE_RATE_HZ)

fig, axes = plt.subplots(2, 1, figsize=(11, 7))
axes[0].plot(t[:TIME_PLOT_SAMPLES] * 1000, strain[:TIME_PLOT_SAMPLES], 'steelblue', lw=1)
axes[0].set_xlabel('Time (ms)'); axes[0].set_ylabel('Strain (με)')
axes[0].set_title('Dynamic Strain (first 400 ms)')
axes[0].grid(True, linestyle=':', alpha=0.5)

axes[1].semilogy(f_pos, mag, 'steelblue', lw=1)

# Annotate all three modal peaks with arrows; text offsets tuned to avoid overlap.
# Each entry: (frequency [Hz], label, color, (label_x [Hz], peak_height_multiplier))
annotations = [
    (47.0,  'Mode 1: 47 Hz',  'tomato',   (70,  8)),
    (87.3,  'Mode 2: 87 Hz',  'seagreen', (150, 3)),
    (312.5, 'Mode 3: 312 Hz', 'purple',   (360, 4)),
]
for freq, label, color, (label_x, height_mult) in annotations:
    idx = np.argmin(np.abs(f_pos - freq))
    peak = mag[idx]
    axes[1].annotate(label,
                     xy=(freq, peak),
                     xytext=(label_x, peak * height_mult),
                     fontsize=9, color=color,
                     arrowprops=dict(arrowstyle='->', color=color, lw=1.5))

axes[1].set_xlabel('Frequency (Hz)'); axes[1].set_ylabel('Amplitude (με)')
axes[1].set_title('FFT')
axes[1].set_xlim(0, 600)
axes[1].grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig32_nb1_dynamic_strain_fft.png', bbox_inches='tight', dpi=150)
plt.show()

## Dynamic Amplification Factor (DAF)

The **dynamic amplification factor** quantifies how much larger the dynamic response is compared to the equivalent static response. For a single-degree-of-freedom system driven harmonically at frequency ratio $r = f/f_n$:

$$\text{DAF} = \frac{1}{\sqrt{(1 - r^2)^2 + (2\zeta r)^2}}$$

where $\zeta$ is the **damping ratio**. At resonance ($r = 1$), the DAF reduces to $1/(2\zeta)$ — light damping ($\zeta = 0.01$) produces amplifications exceeding 50×. This is why resonance avoidance is a primary concern in dynamic strain measurement: a strain gage measuring near resonance will read a signal far larger than the static equivalent, and the result is only meaningful if the dynamic context is understood.

In [ ]:
# --- Dynamic amplification factor vs. frequency ratio, for several damping ratios ---
FREQ_RATIO = np.linspace(0, 3, 1000)        # r = f / f_n
DAMPING_RATIOS = [0.01, 0.05, 0.10, 0.20]   # zeta
Y_LIMIT = 55                                # spans the lightly damped peak, 1/(2*0.01) = 50x


def dynamic_amplification_factor(r, zeta):
    """DAF of a harmonically forced single-DOF system at frequency ratio r = f/f_n.

    Returns 1 / sqrt((1 - r**2)**2 + (2*zeta*r)**2). At resonance (r = 1) this
    reduces to 1/(2*zeta), the quality factor Q.
    """
    return 1 / np.sqrt((1 - r**2)**2 + (2 * zeta * r)**2)


# Each damping ratio gets its own line style as well as a colour, so the
# figure reads in black-and-white.
LINESTYLES = ['-', '--', '-.', ':']
plt.figure()
for zeta, ls in zip(DAMPING_RATIOS, LINESTYLES):
    daf = dynamic_amplification_factor(FREQ_RATIO, zeta)
    plt.plot(FREQ_RATIO, daf, lw=2, ls=ls, label=f'ζ={zeta:.2f}  (peak {1/(2*zeta):g}×)')
plt.axvline(1, color='black', lw=0.8, linestyle=':')
plt.xlabel('r = f/fn'); plt.ylabel('DAF'); plt.title('Dynamic Amplification Factor')
plt.legend(fontsize=9); plt.ylim(0, Y_LIMIT); plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / 'fig32_nb1_dynamic_amplification_factor.png', bbox_inches='tight', dpi=150)
plt.show()

## Where to take this next

These two figures are a starting point. A few concrete extensions, each tied to a concern raised in Chapter 32:

- **Window before you transform.** Apply a Hann window to `strain` before the FFT and compare the spectral peaks with the un-windowed result. The chapter warns that truncating a record before ring-down is complete introduces *spectral leakage*; windowing is the standard defense, and here you can watch the noise skirts around each modal peak narrow.
- **Estimate damping from the peak.** Use the half-power (−3 dB) bandwidth method on a resonance peak to back out the damping ratio ζ, then check it against Q = 1/(2ζ) from Eq. 32.2. Add a lightly damped resonance to the simulated signal so the peak is sharp enough to measure.
- **Watch a peak fall between samples.** Simulate a short half-sine impact (rise time ~0.5 ms, so content to ≈1 kHz) and record its peak strain at sample rates from Nyquist (2 kHz) up to 20× oversampling. Plot the recorded peak versus sample rate to reproduce the chapter's argument that undersampling silently underestimates peak strain — an unconservative error in shock testing.